In [72]:
from ollama import chat
from ollama import ChatResponse
from loguru import logger
import time
from pydantic import BaseModel
from typing import List, Optional

In [ ]:
class ImageDescription(BaseModel):
    people_number: int
    people_gender: Optional[List[str]]
    people_age_range: Optional[List[str]]
    people_mood: Optional[List[str]]
    people_dressing_style: Optional[List[str]]
    overall_mood: Optional[List[str]]

schema = ImageDescription.model_json_schema()

example = ImageDescription(
    people_number=3,
    people_gender=["male", "female"],
    people_age_range=["20-30", "30-40", "40-50"],
    people_mood=["happy", "neutral", "sad"],
    people_dressing_style=["casual", "formal", "sporty"],
    overall_mood=["festive"]
)

In [74]:
SYSTEM_PROMPT = """You are a helpful assistant that describes the people in the image.

Rules:
1. Extract the number of people and their characteristics.
2. Provide a detailed description of each person, including their appearance, clothing, and any notable features.
3. Mention the overall mood or atmosphere conveyed by the people in the image.

Schema:
{schema}

Example:
{example}
"""


In [75]:
QUESTION = "Describe the people in the image. Extract the information according to the schema and rules provided in the system prompt."

# VLM call

In [76]:
VLM_MODEL = 'qwen3-vl:2b'

In [77]:
start_time = time.time()
raw_response: ChatResponse = chat(
	model=VLM_MODEL,
	messages=[{ 'role': 'system', 
                'content': SYSTEM_PROMPT.format(schema=schema, example=example)
                },
                {
                    'role': 'user',
                    'content': QUESTION,
                    'images': ['imgs/car.jpeg']
                },
                ],
        format='json',
        options={"temperature": 0}
            )

logger.info(raw_response['message']['content'])
response = ImageDescription.model_validate_json(raw_response['message']['content'])
elapsed_time = time.time() - start_time
logger.info(f"Elapsed time: {elapsed_time:.2f} seconds")

2026-06-13 18:06:51.314 | INFO     | __main__:<module>:17 - 
{

"people_number": 2,

"people_age_range": ["60-70", "50-60"],

"people_mood": ["happy", "relaxed"],

"people_dressing_style": ["casual", "warm clothing", "jacket", "scarf", "hat"],

"overall_mood": ["relaxed", "happy"]

}
2026-06-13 18:06:51.316 | INFO     | __main__:<module>:20 - Elapsed time: 14.38 seconds
